# Infer independent context and population scores (Production)

Production inference workflow for the V1 representation learning model. Evaluates normal context consistency with EMA latent prediction ($S_{\mathrm{pred}}$) and normal population distance ($S_{\mathrm{pop}}$).

Key features:

- Direct parameters: configure inference directly in the notebook via `InferenceParams` class arguments (e.g. `params = InferenceParams(batch_size=64, mad_multiplier=2.5, in_memory=True)`).
- Manual paths: run from the repository root and set `SRC_DIR`, `V1_DATA_ROOT`, and `V1_CHECKPOINT_PATH` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.
- In-memory caching & streaming: set `in_memory=True` to preload samples into Host RAM for fastest evaluation, or `in_memory=False` for $\mathcal{O}(1)$ disk streaming on $100\text{K}+$ datasets.
- Progress tracking: integrated `tqdm` progress bars for batch scoring across validation and test splits.
- Production checkpoint: restores the trained model weights and normal reference bank from `checkpoints/v1_representation_20260904_01.pt` (override with `V1_CHECKPOINT_PATH`). Inference always fails fast with `FileNotFoundError` when the configured checkpoint is absent and never scores with random weights.
- Compute device: automatically uses CUDA when available (`torch.cuda.is_available()`), with transparent CPU fallback.
- Sharded dataset: reads persisted train, validation, and test shards through `FileDataset`.
- Normal-only reference bank: the restored checkpoint bank is used as-is by default (set `V1_REFIT_BANK=true` to refit on the first train batch); the bank source is printed and recorded with `S_pop`. Validation and test labels are not used for fitting.
- Independent scoring: computes file-level scores $S_{\mathrm{pred}}$ and $S_{\mathrm{pop}}$, independent MAD thresholds, and timestep-level anomaly localization.
- Evaluation: assesses anomaly detection performance against test split ground truth when anomaly metadata is present.


In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from itertools import islice
import json
import os
from pathlib import Path
import sys
import torch
from tqdm.auto import tqdm

# ---- Server paths: repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V1_REPO_ROOT to the checkout path.
V1_REPO_ROOT = os.environ.get('V1_REPO_ROOT', '/home/trietlm/anomaly-representation-learning')
SRC_DIR = os.environ.get('V1_SRC_DIR', str(Path(V1_REPO_ROOT) / 'src'))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / 'representation').is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'representation' package. "
        f"Run from the repository root or set V1_REPO_ROOT / SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded representation modules from: {_source_dir}")

from representation import V1Config
from representation.checkpoint import load_checkpoint, save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference, mad_threshold, prepare_reference_bank
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Hardware] Compute device:', device)
if device.type == 'cuda':
    print('[Hardware] CUDA device name:', torch.cuda.get_device_name(0))
    print('[Hardware] Allocated memory:', f"{torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")


In [ ]:
# ---- Server paths: dataset and checkpoint (edit these one-line values; used exactly) ----
V1_DATA_ROOT = os.environ.get('V1_DATA_ROOT', '/home/trietlm/anomaly-representation-learning/data/generated/production')
V1_CHECKPOINT_PATH = os.environ.get('V1_CHECKPOINT_PATH', os.environ.get('V1_CHECKPOINT', '/home/trietlm/anomaly-representation-learning/checkpoints/v1_representation_20260904_01.pt'))

_default_data, _default_ckpt = V1_DATA_ROOT, V1_CHECKPOINT_PATH

# Opt-in only: refit the reference bank on the first train batch instead of
# using the restored checkpoint bank as-is (default). Recorded in outputs.
REFIT_REFERENCE_BANK = os.environ.get('V1_REFIT_BANK', 'false').lower() in ('true', '1', 'yes')

@dataclass
class InferenceParams:
    """Unified inference configuration (works from a repository checkout).
    
    Modify parameters directly here or pass keyword arguments to InferenceParams(...).
    """
    # Dataset and paths (auto-resolved based on environment)
    data_root: str = _default_data
    checkpoint_path: str = _default_ckpt
    max_samples: int | None = int(os.environ['V1_MAX_SAMPLES']) if 'V1_MAX_SAMPLES' in os.environ else None
    
    # In-memory RAM caching option
    in_memory: bool = os.environ.get('V1_IN_MEMORY', 'true').lower() in ('true', '1', 'yes')
    
    # Inference hyperparameters (optimized defaults for GPU)
    batch_size: int = int(os.environ.get('V1_BATCH_SIZE', '64'))
    mad_multiplier: float = float(os.environ.get('V1_MAD_MULTIPLIER', '2.5'))

# Direct instantiation: edit parameters here directly when running interactively
params = InferenceParams()

configured_root = Path(params.data_root).expanduser()
DATA_ROOT = configured_root if configured_root.is_absolute() else Path.cwd() / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Set V1_DATA_ROOT to the materialized dataset root (local alternative: data/generated/production).")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")

def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples

train_samples = load_split('train')
val_samples = load_split('val', limit=2)
test_samples = load_split('test', limit=2)
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples], 'test IDs', [sample.file_id for sample in test_samples])
print(f"Configured parameters: batch_size={params.batch_size}, mad_multiplier={params.mad_multiplier}, in_memory={params.in_memory}")


In [ ]:
configured_ckpt = Path(params.checkpoint_path).expanduser()
checkpoint_path = configured_ckpt if configured_ckpt.is_absolute() else Path.cwd() / configured_ckpt

# Full inference never runs on random weights: the pinned checkpoint is mandatory.
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"V1 checkpoint not found at {checkpoint_path}. Set V1_CHECKPOINT_PATH to the trained checkpoint file (v1_representation_20260904_01.pt).")
payload = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
saved_cfg = payload.get('config', {})
cfg = V1Config(**saved_cfg)
print(f"Loaded configuration from {checkpoint_path}: d_model={cfg.d_model}, layers={cfg.sequence_layers}, heads={cfg.attention_heads}")

patchifier = Patchifier(PatchConfig(patch_size=cfg.patch_size, stride=cfg.stride, pad_end=True))

class StreamingBatchDataset:
    """Yield collated minibatches, with optional Host RAM caching and full shuffling."""
    def __init__(self, data_root, split, patchifier, config, b_size=64, max_count=None, base_seed=0, in_memory=True):
        self.data_root = data_root
        self.split = split
        self.patchifier = patchifier
        self.config = config
        self.batch_size = b_size
        self.max_count = max_count
        self.base_seed = base_seed
        self.in_memory = in_memory
        self.samples = None
        
        if in_memory:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            total = manifest['counts'][split] if self.max_count is None else min(self.max_count, manifest['counts'][split])
            self.samples = list(tqdm(iterator, total=total, desc=f"Loading {split} to RAM"))

    def __iter__(self):
        if self.samples is not None:
            chunk = []
            batch_idx = 0
            for sample in self.samples:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )
        else:
            dataset = FileDataset(self.data_root, split=self.split)
            iterator = islice(dataset, self.max_count) if self.max_count is not None else iter(dataset)
            chunk = []
            batch_idx = 0
            for sample in iterator:
                chunk.append(sample)
                if len(chunk) == self.batch_size:
                    yield collate_variable_files(
                        chunk,
                        self.patchifier,
                        masking_config=self.config,
                        masking_seed=self.base_seed + batch_idx,
                    )
                    chunk = []
                    batch_idx += 1
            if chunk:
                yield collate_variable_files(
                    chunk,
                    self.patchifier,
                    masking_config=self.config,
                    masking_seed=self.base_seed + batch_idx,
                )

reference_batches = StreamingBatchDataset(DATA_ROOT, 'train', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=5, in_memory=params.in_memory)
val_batches = StreamingBatchDataset(DATA_ROOT, 'val', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=6, in_memory=params.in_memory)
test_batches = StreamingBatchDataset(DATA_ROOT, 'test', patchifier, cfg, b_size=params.batch_size, max_count=params.max_samples, base_seed=7, in_memory=params.in_memory)

reference_batch = next(iter(reference_batches))
val_batch = next(iter(val_batches))
test_batch = next(iter(test_batches))
print('reference signals', tuple(reference_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'test signals', tuple(test_batch['signals'].shape))


In [ ]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
model.to(device)
model.eval()

bank = NormalReferenceBank(k=min(cfg.knn_k, manifest['counts']['train']))
if checkpoint_path.is_file():
    meta = load_checkpoint(checkpoint_path, model, reference_bank=bank)
    print(f"Restored checkpoint from {checkpoint_path} (step {meta['step']}, reference bank restored: {meta['has_reference_bank']})")

ref_batch_device = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in reference_batch.items()}
with torch.no_grad():
    reference_output = model(ref_batch_device)
bank_source = prepare_reference_bank(bank, reference_output['file_embedding'], refit=REFIT_REFERENCE_BANK)

inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)

scored_splits = {}
for split, batch_loader in (('val', val_batches), ('test', test_batches)):
    all_pred = []
    all_pop = []
    all_is_anomalous = []
    all_families = []
    sample_timesteps = []
    
    total_files = len(batch_loader.samples) if batch_loader.samples is not None else (
        manifest['counts'][split] if params.max_samples is None else min(params.max_samples, manifest['counts'][split])
    )
    total_b = (total_files + params.batch_size - 1) // params.batch_size
    pbar = tqdm(batch_loader, total=total_b, desc=f"Scoring {split} ({total_files} files)")
    for batch in pbar:
        scores = inference.score_batch(batch)
        all_pred.extend(scores['S_pred'].cpu().tolist())
        all_pop.extend(scores['S_pop'].cpu().tolist())
        if 'file_labels' in batch and batch['file_labels'] is not None:
            all_is_anomalous.extend([getattr(l, 'value', str(l)).lower() == 'abnormal' for l in batch['file_labels']])
        if 'anomaly_meta' in batch and batch['anomaly_meta'] is not None:
            all_families.extend([meta.family.value if meta is not None else 'normal' for meta in batch['anomaly_meta']])
        if len(sample_timesteps) < 5:
            sample_timesteps.extend(scores['timestep_scores'][:5 - len(sample_timesteps)])
            
    pred_t = torch.tensor(all_pred)
    pop_t = torch.tensor(all_pop)
    t_pred = mad_threshold(pred_t, multiplier=params.mad_multiplier)
    t_pop = mad_threshold(pop_t, multiplier=params.mad_multiplier)
    scored_splits[split] = {
        'S_pred': pred_t,
        'S_pop': pop_t,
        'timestep_scores': sample_timesteps,
        'thresh_pred': t_pred,
        'thresh_pop': t_pop,
        'is_anomalous': all_is_anomalous,
        'families': all_families,
    }
    print(split, 'S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())
    print(f"[{split}] scored {len(all_pred)} files | S_pred: mean={pred_t.mean():.4f}, threshold={t_pred:.4f} | S_pop: mean={pop_t.mean():.4f}, threshold={t_pop:.4f}")

print(f"normal train reference rows {bank.embeddings.shape[0]} (bank source: {bank_source}); labels were not passed to bank.fit")


In [ ]:
# Production anomaly detection performance evaluation on test set
from sklearn.metrics import roc_auc_score

test_anomalous_gt = scored_splits.get('test', {}).get('is_anomalous', [])
n_abnormal = sum(test_anomalous_gt)
n_normal = len(test_anomalous_gt) - n_abnormal
print(f"Ground truth distribution in test set: {n_normal} normal, {n_abnormal} abnormal (total {len(test_anomalous_gt)})")

if n_abnormal > 0 and n_normal > 0:
    test_s_pred = scored_splits['test']['S_pred']
    test_s_pop = scored_splits['test']['S_pop']
    
    # Validation-calibrated thresholds
    val_s_pred = scored_splits.get('val', {}).get('S_pred', test_s_pred)
    val_s_pop = scored_splits.get('val', {}).get('S_pop', test_s_pop)
    th_pred = mad_threshold(val_s_pred, multiplier=params.mad_multiplier)
    th_pop = mad_threshold(val_s_pop, multiplier=params.mad_multiplier)

    det_pred = (test_s_pred > th_pred).tolist()
    det_pop = (test_s_pop > th_pop).tolist()
    det_any = [bool(p or q) for p, q in zip(det_pred, det_pop)]

    tp = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if d and gt)
    fp = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if d and not gt)
    fn = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if not d and gt)
    tn = sum(1 for d, gt in zip(det_any, test_anomalous_gt) if not d and not gt)
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-6, prec + rec)
    
    auc_pred = float(roc_auc_score(test_anomalous_gt, test_s_pred.tolist()))
    auc_pop = float(roc_auc_score(test_anomalous_gt, test_s_pop.tolist()))
    
    print(f"Decision thresholds (multiplier={params.mad_multiplier}): th_pred={th_pred:.4f}, th_pop={th_pop:.4f}")
    print(f"Test classification: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
    print(f"AUROC: S_pred={auc_pred:.4f}, S_pop={auc_pop:.4f}")
    
    # Anomaly family breakdown
    families = scored_splits.get('test', {}).get('families', [])
    if families:
        detected_by_family = {}
        for fam, is_det in zip(families, det_any):
            if fam != 'normal':
                detected_by_family.setdefault(fam, []).append(is_det)
        for fam, det_list in sorted(detected_by_family.items()):
            print(f"Family {fam:28s}: detected {sum(det_list)}/{len(det_list)}")
else:
    print("Test split does not contain both normal and abnormal files; evaluation metrics skipped.")
